# Example 2 — LTE 450 simulation

Simulate the corrected COPEL dataset: 86 clients and 5 cells, each with three 120-degree sectors. Install SDK 0.1.25+ using `prepare_workspace.py`, restart the kernel, and run from the downloaded `notebooks` folder with network and features services available. The project is `../projects/example_2_LTE_simulation`.

All interfaces use 450 MHz. Sector azimuths (0, 120, 240 degrees), transmit powers (43 dBm for cells, 20 dBm for clients), and the LTE metric are inherited from example 1. Antenna gains are assumed constants: 11 dBi for sectors and 0 dBi for clients. Sector beamwidth filters candidates; no measured 450 MHz antenna patterns are used. CSV mounting heights override the antenna defaults. These settings illustrate the workflow and should be replaced with equipment parameters for an operational study.

The imported coordinates were repaired and cross-checked in EPSG:31982. The candidate step uses all 91 sites; metric computation may request hundreds of geographic links on the first run. Raw features are cached in this project for subsequent runs.

For the full run, configure the network service with `PLANNING_WORKER_REQUEST_TIMEOUT=1800` and restart it before opening a session. This notebook waits up to 1900 seconds for planning responses and 300 seconds for individual geographic requests.

This project now uses an exploratory bidirectional LTE link-margin metric, embedded in scenario.toml. The editable reference is ../metrics/lte450_link_margin.toml. Required receiver powers (-110 dBm client / -115 dBm cell), 3 dB system loss, 6 dB clutter allowance, and 10 dB fade margin are assumptions. With rank_threshold=2, a good direct link meets both assumed budgets after reserves. This is not a calibrated throughput or interference model.


## 1. Create possible connections

Open the example and see which connections are available.


In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject, plot_network

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1900.0)
project = PlanningProject("../projects/example_2_LTE_simulation")
scenario = client.open_project(project, planner="cell")
candidates = scenario.build_candidate_edges()
plot_network(scenario.export_result(), kind="candidate", save_path=project.path / "candidate_network.html");


## 2. Explore possible connections

Type or select an endpoint to show its incoming and outgoing connections. Select a connection from the filtered list, or use Previous and Next. Clear the endpoint to show all links. Choose what to inspect with the Show menu. The view updates inside the panel.


In [ ]:
from cisei_features_sdk.notebook import FeaturesNotebook

geo = FeaturesNotebook("http://features:8080")
candidate_links = scenario.get_candidate_links()
candidate_panel = scenario.show_link_browser(geo, candidate_links)


## 3. Calculate connection quality

Calculate the quality of each possible connection.


In [ ]:
import os

metrics = scenario.compute_metrics(
    geo_base_url=os.getenv("PLANNING_GEO_URL", "http://features:8080"),
    geo_timeout=300.0,
    batch_size=20,
    on_batch=lambda progress: print(f"Saved features: {progress['next_offset']}/{progress['total_links']} links processed"),
)
scenario.show_metrics()


## 4. Choose the network


In [ ]:
solution = scenario.solve()


## 5. Check quality and save

Review coverage, connections, and cell loads using a maximum accumulated cost of 2. Save the network together with this quality report.


In [ ]:
scenario.show_evaluation(rank_threshold=2)
scenario.save_result(rank_threshold=2)


## 6. Open the saved network

This view reads the saved file. Exploring chosen connections below still uses the live planning session.


In [ ]:
saved_result = scenario.load_result()
plot_network(saved_result, kind="final", save_path=project.path / "solution_network.html");


## 7. Explore chosen connections

Use the same controls to inspect the connections selected for the network.


In [ ]:
solution_links = scenario.get_solution_links()
solution_panel = scenario.show_link_browser(geo, solution_links)


## 8. Finish

Release the map and profile worker when you are done. Your saved network remains in the project.


In [ ]:
geo.close_worker();
